# Merge dates for one site

## In short

The batch notebooks make one CSV for each flight date. This notebook joins all the date CSVs for one site into one table (the site master CSV). It also adds columns for the site, the date, and the image type.

To use it, set the input and output folders at the top of the code cell. Then run the cell.

## Details

**Input**

The per-date CSVs from the batch notebooks. The file names must have this format:

```
<site>_<YYYY-MM-DD>_MS_VI.csv
<site>_<YYYY-MM-DD>_RGB_VI.csv
```

The notebook reads the site, the date, and the image type from the file name.

**Output**

`output_folder/<site>_<image type>_master.csv`

**Added columns** (at the start of the table)

| Column | Value |
|---|---|
| `Site` | The site from the file name, or `site_name_override` |
| `Date` | The date from the file name (`YYYY-MM-DD`) |
| `Image_Type` | `MS` or `RGB` |
| `Missing_val` | The number of empty values in that row |

**How to run**

1. Set `input_folder` to the folder with the per-date CSVs.
2. Set `output_folder`.
3. Set `image_type_label` to `"MS"` or `"RGB"`.
4. To give the site a different name, set `site_name_override`. This step is optional. Use it to make the site name the same in the RGB and MS data.
5. Run the cell.
6. Check the printout. Make sure that each date has the correct number of rows.

**Notes**

- Run the notebook one time for each site and each image type.
- The plot grid shapefile can have a column with the same name as an index (for example `NDVI`). In that case, pandas changes the name of the calculated index column when it reads the CSV (for example to `NDVI.1`). Check the column names before you use the data.
- Empty values are written as `nan`.


In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# USER SETTINGS: change these when working on a new site
# ============================================================

# Folder with the per-date CSVs from the batch notebook.
input_folder = Path('path/to/output/MSorRGB')

# The site master CSV is written here.
output_folder = Path('path/to/merged_output')

# "MS" or "RGB". Selects files named *_MS_VI.csv or *_RGB_VI.csv.
image_type_label = "MS"

# None takes the site name from the file names.
# Set a name (for example "RoujolA") to use that name instead.
site_name_override = None

# ============================================================
# 1. Check folders
# ============================================================

print("Input folder exists:", input_folder.exists())
print("Output folder exists before mkdir:", output_folder.exists())

output_folder.mkdir(parents=True, exist_ok=True)

print("Output folder exists after mkdir:", output_folder.exists())

# ============================================================
# 2. Find all VI CSV files for this image type
# ============================================================

csv_files = sorted(input_folder.glob(f"*_{image_type_label}_VI.csv"))

print(f"\nFound {len(csv_files)} {image_type_label} CSV files:")

for file in csv_files:
    print(file.name)

# Stop early if no files are found
if len(csv_files) == 0:
    raise FileNotFoundError(f"No {image_type_label} CSV files found. Check your input_folder path or filename pattern.")

# ============================================================
# 3. Read each CSV, add metadata, reorder columns
# ============================================================

all_data = []

for file in csv_files:
    df = pd.read_csv(file)
    
    # Example filename:
    # RoujolA_2024-07-24_MS_VI.csv
    #
    # Safer parsing from the right:
    # site = everything before date
    # date = third from end
    # image_type = second from end
    parts = file.stem.split("_")

    site = "_".join(parts[:-3])
    date = parts[-3]
    image_type = parts[-2]

    df["Site"] = site_name_override or site
    df["Date"] = date
    df["Image_Type"] = image_type
    df["Missing_val"] = df.isna().sum(axis=1)

    # Keep metadata columns at the beginning
    front_cols = ["Site", "Date", "Image_Type", "Missing_val"]

    other_cols = [col for col in df.columns if col not in front_cols]
    df = df[front_cols + other_cols]

    all_data.append(df)

print(f"\nPrepared {len(all_data)} dataframes.")

# ============================================================
# 4. Preview processed files before combining
# ============================================================

print("\nPreview first processed dataframe:")
display(all_data[0].head(10))

print("\nSummary of each processed dataframe:")

for i, df in enumerate(all_data):
    print("Dataframe number:", i)
    print("Site:", df["Site"].iloc[0])
    print("Date:", df["Date"].iloc[0])
    print("Image type:", df["Image_Type"].iloc[0])
    print("Shape:", df.shape)
    print("Total missing values:", df["Missing_val"].sum())
    print("-" * 50)

# ============================================================
# 5. Show rows with missing values before combining
# ============================================================

print("\nRows with missing values by file:")

for i, df in enumerate(all_data):
    missing_rows = df[df["Missing_val"] > 0]

    if len(missing_rows) > 0:
        print("Dataframe number:", i)
        print("Date:", df["Date"].iloc[0])
        print("Rows with missing values:", len(missing_rows))
        display(missing_rows.head())
        print("-" * 50)

# ============================================================
# 6. Combine all files into one master dataframe
# ============================================================

master_df = pd.concat(all_data, ignore_index=True)

print(f"\n{image_type_label} master dataframe created.")
print("Shape:", master_df.shape)

display(master_df.head())
display(master_df.tail())

# ============================================================
# 7. Quality checks
# ============================================================

print("\nRows per date:")
display(master_df["Date"].value_counts().sort_index())

print("\nMissing values by date:")
display(
    master_df
    .groupby("Date")["Missing_val"]
    .sum()
    .sort_values(ascending=False)
)

print("\nRows per site:")
display(master_df["Site"].value_counts())

# ============================================================
# 8. Save site master CSV
# ============================================================

site_name = master_df["Site"].iloc[0]

output_file = output_folder / f"{site_name}_{image_type_label}_master.csv"

master_df.to_csv(output_file, index=False, na_rep="nan")

print(f"\nSaved {image_type_label} master CSV to:")
print(output_file)